# 第13回：Kaggleに入って最初の提出を作る

**今日の問い：コンペの説明を、ローカルの分析手順へどう翻訳するか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 問題・指標・データ・提出形式を読み解き、再現可能なベースラインを作る
- cross_val_predictでOOF予測を作り、CVとLBの一致を確かめる
- 提出CSVを検査する関数を、テスト付きで書く

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- Leaderboard：提出結果を順位表示する仕組み
- OOF予測：交差検証の検証側だけを集めた予測
- submission：指定形式の予測ファイル
- CV-LBギャップ：手元の検証と公開スコアの差
- ベースライン：最初に必ず保存する比較起点

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## コンペは「これまでの総合演習」

Kaggle（や、この教材のローカル模擬コンペ）は、第1〜12回で学んだことを1本の流れにする総合演習です。
新しい魔法はありません。むしろ大事なのは、**コンペの説明を、いつもの分析手順へ翻訳する**こと。

最初に必ず4点を確認します：**目的（何を予測）／評価指標／データ（trainとtestの違い）／提出形式**。
まずデータを開いて形を見ます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
sample = pd.read_csv(DATA / "local_competition" / "sample_submission.csv")
print("train:", train.shape, "test:", test.shape, "提出見本:", sample.shape)
display(train.head(3))
display(sample.head(3))


### 出力の読み方

- **trainには`active`列があり、testには無い**はずです。testの答えは伏せられていて、提出して初めて採点されます。
- **提出見本(sample_submission)**は「こういう形で出してね」という雛形。列名と行数を必ずこれに合わせます。
- trainとtestの行数を足すと、第3回で見た元データの件数に対応します。


## コンペ説明（この模擬コンペの4点）

- **目的**：実験計画時の情報から活性`active`（0/1）を予測する
- **指標**：F1（第8回。活性が少ないのでaccuracyでなくF1）
- **データ**：`train.csv`には答えあり、`test.csv`には無し
- **提出形式**：`sample_id`と`active`の2列

Kaggle Titanicを使う場合も、最初にこの4点（目的・指標・train/test・提出形式）を同じように確認します。


## ベースラインを作る（第9回のPipelineを再利用）

第9回で学んだ`ColumnTransformer`＋`Pipeline`をそのまま使い、数値もカテゴリも安全に1つのモデルへ通します。
`sample_id`や実験後の列など、**使ってはいけない列を`drop_columns`で外す**のがポイント（第5回のリーク回避）。
まずローカルの検証F1で当たりを付けます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

target = "active"
drop_columns = ["sample_id", "experiment_date", "smiles", target]
features = [column for column in train.columns if column not in drop_columns]
numeric = train[features].select_dtypes(include="number").columns.tolist()
categorical = [column for column in features if column not in numeric]
preprocess = ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
model = Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))])
X_train, X_valid, y_train, y_valid = train_test_split(train[features], train[target], test_size=0.25, random_state=42, stratify=train[target])
model.fit(X_train, y_train)
print("ローカル検証F1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


### 出力の読み方

このローカル検証F1が、あなたの**最初のものさし**です。以降の改善は、必ずこの値と比べます。
「提出して順位が上がったか」だけでなく、**手元の検証がどう動いたか**を先に見る習慣が、コンペで
崩れないコツです（次のDEEP DIVEのCV-LBの話につながります）。


## TRY：提出CSVを作り、機械的に検査する

提出でいちばん多い失敗は、モデルの精度ではなく**フォーマットのミス**（列名・行数・余計なindex列）。
`assert`で自動チェックしてから保存します。`index=False`で余計な行番号列を混ぜないことも重要です。


In [ ]:
model.fit(train[features], train[target])
submission = pd.DataFrame({"sample_id": test["sample_id"], "active": model.predict(test[features])})
assert list(submission.columns) == ["sample_id", "active"]
assert len(submission) == len(test)
assert submission["sample_id"].is_unique
output = ROOT / "workspace" / "submission_baseline.csv"
submission.to_csv(output, index=False)
print("保存先:", output)
submission.head()


### 出力の読み方

3つの`assert`を通ってCSVが保存されれば、形式は合格。`workspace/`に出力されるので、Kaggleが使える人は
これをアップロードします。使えない場合は、講師がローカルで採点します（第14回）。

## CHANGE

提出前に変えるのは**1点だけ**（第12回の原則）。例：`max_depth=6`を`3`へ変え、ローカル検証F1がどう動くか
確認してから提出します。


## DEEP DIVE：手元でLeaderboardを予想する（OOF）と、提出を守る

コンペで沼にはまる典型が「提出回数を無駄遣いして、手元で何も分かっていない」状態です。
**OOF予測**で手元にLeaderboard相当の推定を持ち、**提出バリデータ**で形式ミスを防ぎます。


### OOF予測：提出せずにスコアを見積もる

`cross_val_predict`は、各行を「その行を学習に使っていないモデル」で予測します（OOF＝out-of-fold）。
これを全部集めれば、**提出しなくても**手元でLeaderboardに近いF1を推定できます。提出回数の節約になります。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score

oof = cross_val_predict(model, train[features], train[target], cv=StratifiedKFold(5, shuffle=True, random_state=42))
print("OOF F1:", round(f1_score(train[target], oof), 3))
print("この値は、公開スコアの当たりを付ける手元の推定として使える。")


### 出力の読み方

このOOF F1と、実際に提出したときのスコア（LB）を比べます。**両者が近ければ**手元の検証は信頼でき、
改善の判断を手元だけで進められます。**大きく食い違えば**、分布ずれ（第6回のadversarial validation）や
リークを疑います。この差を**CV-LBギャップ**と呼びます。


### 提出バリデータを「テスト」する

第2回で学んだ「テストで守る」を提出に適用します。検査関数を書くだけでなく、**わざと壊した提出**を
渡して、すべての`assert`がちゃんと弾くかを確かめます。関数が本当に機能する保証になります。


In [ ]:
def validate_submission(sub, test, expected=("sample_id", "active")):
    "提出CSVの列・行数・ID一致・値域を検査する。問題があればAssertionError。"
    expected = list(expected)
    assert list(sub.columns) == expected, "列名または順序が違います"
    assert len(sub) == len(test), "行数がtestと一致しません"
    assert sub[expected[0]].is_unique, "IDが重複しています"
    assert sub[expected[0]].tolist() == test[expected[0]].tolist(), "IDの順序がtestと一致しません"
    assert sub[expected[1]].isin([0, 1]).all(), "予測値は0/1にしてください"
    return "提出形式OK"

print(validate_submission(submission, test))
broken = submission.copy()
broken.loc[broken.index[0], "active"] = 5
try:
    validate_submission(broken, test)
except AssertionError as error:
    print("異常を検出:", error)


### 出力の読み方

正しい提出は「提出形式OK」を返し、`active`に5を混ぜた壊れた提出は「異常を検出: 予測値は0/1に…」で
弾かれます。**弾かれることを確認して初めて**、検査関数は信頼できます。本番の提出前に必ず通す関数として
手元に残しておきましょう。


## よくある誤り

- testの情報へ合わせて特徴量を決める
- 提出ファイルのindex列を混入させる
- 最初から公開Notebookを丸ごと写す

## SELF-STUDY（任意・30〜60分）

- OOFのF1と提出後スコアの差を記録し、原因を1つ推測する
- validate_submissionへ異常な提出を渡し、全assertが働くか試す

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. OOF予測は何に使えるか
2. CV-LBギャップが大きいとき何を疑うか
3. 提出前に検査する項目は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
